# 06주차 · 임베딩 기반 분류·군집·시각화

**임베딩 기반 데이터 과학 — 국립목포대학교 컴퓨터학부 4학년**

이 Notebook은 *Hands-On Large Language Models* 공개 저장소의 개념과 실습 흐름을
한국어 수업에 맞게 새로 구성한 파생 강의자료입니다. 원본은 Apache License 2.0을
따르며, 출처와 변경 사항은 `SOURCE_AND_LICENSE.md`에 기록했습니다.

- 원본: https://github.com/HandsOnLLM/Hands-On-Large-Language-Models
- 기준 커밋: `ea3390819997999a51983677b80b3aac4dc50ada`
- 권장 환경: Google Colab 또는 Python 3.11+


## 학습목표

- 임베딩을 머신러닝 특징으로 사용한다.
- 분류와 군집의 평가 방법을 구분한다.
- 차원축소 그림을 원래 공간의 완전한 복사로 오해하지 않는다.


In [ ]:
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / denom) if denom else 0.0


In [ ]:
texts = [
    "청년 창업 사업비 지원", "관광 창업 기업 공간 제공", "예비 창업자 창업 교육",
    "노인 복지 이동 차량 운영", "고령자 복지 돌봄 서비스", "장애인 복지 이동 지원",
    "해양 관광 코스 개발", "섬 관광 콘텐츠 제작", "지역 축제 관광 홍보",
]
labels = ["창업"]*3 + ["복지"]*3 + ["관광"]*3
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(2, 4))
X = vectorizer.fit_transform(texts)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import classification_report
clf = LogisticRegression(C=10, max_iter=1000, random_state=SEED)
pred = cross_val_predict(clf, X, labels, cv=LeaveOneOut())
print(classification_report(labels, pred, zero_division=0))
classification_results = pd.DataFrame({"문서": texts, "정답": labels, "예측": pred})
classification_results["정답여부"] = classification_results["정답"] == classification_results["예측"]
classification_results.loc[~classification_results["정답여부"]]


In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
km = KMeans(n_clusters=3, random_state=SEED, n_init=20)
cluster = km.fit_predict(X)
Z = TruncatedSVD(n_components=2, random_state=SEED).fit_transform(X)
result = pd.DataFrame({"문서": texts, "정답": labels, "분류예측": pred, "군집": cluster, "x": Z[:,0], "y": Z[:,1]})
result


## 분류와 군집의 평가 기준은 다르다

분류는 정답 라벨과 예측을 비교한다. 군집의 실루엣 계수는 벡터 공간 내부의 응집도·분리도를 측정하고, ARI는 정답 라벨이 있을 때 군집과 외부 기준의 일치 정도를 측정한다. 높은 실루엣 계수가 곧 유용한 주제를 뜻하지는 않는다.


In [ ]:
from sklearn.metrics import silhouette_score, adjusted_rand_score
cluster_metrics = pd.Series({
    "silhouette_cosine": silhouette_score(X, cluster, metric="cosine"),
    "adjusted_rand_index": adjusted_rand_score(labels, cluster),
})
cluster_metrics.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for _, r in result.iterrows():
    ax.scatter(r.x, r.y, c=f"C{int(r.군집)}")
    ax.annotate(r.문서[:7], (r.x, r.y))
ax.set_title("임베딩의 2차원 투영")
plt.show()


## 학생 활동

- 문서 수를 범주별 10개 이상으로 늘려라.
- 분류 실패 문서와 군집 경계 문서를 비교하라.
- PCA/SVD 그림만 보고 군집 품질을 판단하면 안 되는 이유를 작성하라.


---
## 학습 기록과 생성형 AI 사용 내역

다음 항목을 자신의 말로 작성하세요.

1. 이번 실습에서 가장 중요한 결과는 무엇인가?
2. 결과를 뒷받침하는 수치 또는 그래프는 무엇인가?
3. 실패하거나 예상과 달랐던 부분은 무엇인가?
4. 생성형 AI를 사용했다면 프롬프트, 채택·거부한 제안, 직접 검증한 내용을 기록하라.
